# Vantara Retail Solutions — Feature Engineering & Leakage Prevention
## Point-in-Time Behavioral, RFM, Velocity & Target Extraction

This notebook documents the feature engineering pipeline with strict point-in-time cutoff discipline (observation period ending 90 days before final date, and target window covering the final 90 days). It evaluates feature correlations with churn, multicollinearity via VIF, and engagement score distributions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.utils.config import ConfigManager

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
%matplotlib inline

### 1. Load Processed Customer Features

In [ ]:
feature_path = ConfigManager.get("paths.customer_features_file", "data/processed/customer_features.parquet")
df = pd.read_parquet(feature_path)
print(f"Loaded feature dataset: {df.shape[0]:,} customers, {df.shape[1]} features/targets")
df.head()

### 2. Target Distributions (Churn & 90-Day Forward CLV)

In [ ]:
churn_counts = df['churn'].value_counts(normalize=True)
print(f"Churn Distribution:\nRetained (0): {churn_counts.get(0, 0):.2%}\nChurned  (1): {churn_counts.get(1, 0):.2%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=df, x='churn', ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('90-Day Churn Class Balance')
axes[0].set_xticklabels(['Retained (0)', 'Churned (1)'])

pos_clv = df[df['clv_next_90d'] > 0]['clv_next_90d']
sns.histplot(np.log1p(pos_clv), bins=30, kde=True, ax=axes[1], color='#3498db')
axes[1].set_title('Forward 90-Day CLV (Log Scale for Retained Customers)')
axes[1].set_xlabel('Log(1 + Forward Spend £)')
plt.show()

### 3. Feature Correlations with Churn

In [ ]:
numeric_df = df.select_dtypes(include=['number'])
churn_corr = numeric_df.corr()['churn'].sort_values()

plt.figure(figsize=(10, 8))
churn_corr.drop(['churn', 'clv_next_90d', 'target_orders_count']).plot(kind='barh', color='#8e44ad')
plt.title('Feature Correlations with 90-Day Churn (Point-in-Time)')
plt.xlabel('Pearson Correlation')
plt.show()

### 4. Engagement Score vs Churn Rate

In [ ]:
tier_churn = df.groupby('engagement_tier')['churn'].agg(count='count', churn_rate='mean').reset_index()
print("Churn Rate by Engagement Health Tier:")
display(tier_churn)

plt.figure(figsize=(8, 4))
sns.barplot(data=tier_churn, x='engagement_tier', y='churn_rate', order=['Low', 'Medium', 'High'], palette='Reds_r')
plt.title('Observed Churn Rate by Engagement Tier')
plt.ylabel('Churn Probability')
plt.ylim(0, 1.0)
plt.show()